The purpose of the models found in this file is to generate images. Initially, this generation will be based soely on attempting to create synthetic images that look similar to those foun in a dataset. Later, prompting through text (both positive and negative) will be added in, and finally, images will also be included. The goal is to create a model that can generate images from scratch, as well as edit images when needed.

PyTorch will be used throughout for both loading datasets and building the model.

In [1]:
# Initial Imports
import os
import random

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from torchvision import datasets, transforms
from einops import rearrange

import matplotlib.pyplot as plt
import numpy as np

In [2]:
# Setting torch to use GPU acceleration if possible.
device = torch.device("cpu")

if torch.cuda.is_available():
    device = torch.device("cuda")

torch.set_default_device(device)
print(f"Using device: {torch.get_default_device()}")

Using device: cpu


#### Model #1 - Unconditional Generation
This model will not have any text prompting or base image to work on, and will instead attmpet to copy a dataset as closely as possible.

In [3]:
# Download and load the dataset.
imageTransform = transforms.Compose([transforms.ToTensor()])
dataset = datasets.CIFAR10(root = "data", train = True, download = True, transform = imageTransform)

In [4]:
# Used as part of training and for removing noise when generating images.
class NoiseScheduler:
    def __init__(self, timesteps = 1000, betaMin = 1e-4, betaMax = 0.02):
        self.timesteps = timesteps
        self.betas = torch.linspace(betaMin, betaMax, steps = timesteps)
        self.alphas = 1.0 - self.betas
        self.cumulativeAlphas = torch.cumprod(self.alphas, dim = 0)

    def add_noise(self, images, timestamp):
        noise = torch.randn_like(images)
        alphaValue = self.cumulativeAlphas[timestamp]
        noisyImages = torch.sqrt(alphaValue) * images + torch.sqrt(1 - alphaValue) * noise
        return noisyImages, noise

In [5]:
# For time, we'll use sinusoidal embeddings.
class SinusoidalEmbeddings(nn.Module):
    def __init__(self, sequenceLength, embeddingDim):
        super(SinusoidalEmbeddings, self).__init__()
        position = torch.arange(0, sequenceLength, dtype = torch.float).unsqueeze(1)
        divTerm = torch.pow(10000.0, (torch.arange(0, embeddingDim, 2, dtype = torch.float) * 2 / embeddingDim))
        self.embeddings = torch.zeros(sequenceLength, embeddingDim, requires_grad = False)
        self.embeddings[:, 0::2] = torch.sin(position * divTerm)
        self.embeddings[:, 1::2] = torch.cos(position * divTerm)

    def forward(self, t):
        return self.embeddings[t]


# Meanwhile for the class, we'll use learnable embeddings.
class LearnableEmbeddings(nn.Module):
    def __init__(self, sequenceLength, embeddingDim):
        super(LearnableEmbeddings, self).__init__()
        self.encoding = nn.Parameter(torch.zeros(sequenceLength, embeddingDim))
        nn.init.xavier_uniform_(self.encoding)

    def forward(self, x):
        return self.encoding[x, :]

In [6]:
# We'll use residual connections to prevent vanishing gradients.
class ResidualBlock(nn.Module):
    def __init__(self, channels, stepEmbeddingSize, classEmbeddingSize):
        super(ResidualBlock, self).__init__()
        self.conv = nn.Conv2d(channels, channels, 3, padding = 1)
        self.bn = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace = True)

        self.linStep = nn.Linear(stepEmbeddingSize, channels)
        self.linClass = nn.Linear(classEmbeddingSize, channels)

    def forward(self, x, stepEmbeddings, classEmbeddings):
        sEmbeddings = self.linStep(stepEmbeddings).unsqueeze(-1).unsqueeze(-1)
        cEmbeddings = self.linClass(classEmbeddings).unsqueeze(-1).unsqueeze(-1)
        
        return self.relu(self.bn(self.conv(x + sEmbeddings + cEmbeddings))) + x


# We'll also use attention layers
class Attention(nn.Module):
    def __init__(self, channels, numHeads):
        super(Attention, self).__init__()
        self.numHeads = numHeads
        self.channels = channels
        self.lin1 = nn.Linear(channels, channels * 3 * numHeads)
        self.lin2 = nn.Linear(channels * numHeads, channels)

    def forward(self, x):
        b, c, h, w = x.shape
        # (B, C, H, W) --> (B, H*W, C). We need channels at the end, since that's what we want the linear layer to operate on.
        x = rearrange(x, 'B C H W -> B (H W) C')
        # (B, H*W, C) --> (B, H*W, C*3)
        x = self.lin1(x)
        # (B, H*W, C) --> (K, B, N, H*W, C). K is what we'll use to get Q, V, and K. N is the number of heads. Use X for H*W.
        x = rearrange(x, 'B X (C N K) -> K B N X C', K = 3, N = self.numHeads)
        # Apply Attention. (K, B, N, H*W, C) --> (B, N, H*W, C)
        Q, K, V = x
        x = F.softmax((Q @ K.transpose(2, 3))/ K.shape[-1], dim = -1) @ V
        # (B, N, H*W, C) -> (B, H, W, C*N).
        x = rearrange(x, 'B N (H W) C -> B H W (C N)', W = w, H = h)
        # (B, H, W, C*N) -> (B, H, W, C)
        x = self.lin2(x)
        # (B, H, W, C) --> (B, C, H, W). Get it back into the form it came in as.
        return rearrange(x, 'B H W C -> B C H W')


# Finally for our actual UNET architecture, we'll need various layers.
class UNETLayer(nn.Module):
    def __init__(self, doesUpscale, hasAttention, channels, stepEmbeddingSize, classEmbeddingSize, numHeads = 0):
        super(UNETLayer, self).__init__()
        self.res1 = ResidualBlock(channels, stepEmbeddingSize, classEmbeddingSize)
        self.res2 = ResidualBlock(channels, stepEmbeddingSize, classEmbeddingSize)

        if doesUpscale:
            self.conv = nn.ConvTranspose2d(channels, channels // 2, kernel_size = 4, stride = 2, padding = 1)

        else:
            self.conv = nn.Conv2d(channels, channels * 2, kernel_size = 3, stride = 2, padding = 1)

        if hasAttention:
            self.attention = Attention(channels, numHeads)

        self.hasAttention = hasAttention

    def forward(self, x, stepEmbeddings, classEmbeddings):
        x = self.res1(x, stepEmbeddings, classEmbeddings)

        if self.hasAttention:
            x = self.attention(x)

        x = self.res2(x, stepEmbeddings, classEmbeddings)
        return self.conv(x), x # Return the raw output as well for the Residual Concatenations later.

In [7]:
class UNET(nn.Module):
    def __init__(self, layerChannels, layerAttention, layerUpscale, inputChannels, outputChannels, numHeads, timeSteps, classCount, stepEmbeddingSize, classEmbeddingSize):
        super(UNET, self).__init__()
        # Encoder Blocks
        self.stepEncoder = SinusoidalEmbeddings(timeSteps, stepEmbeddingSize)
        self.classEncoder = LearnableEmbeddings(classCount, classEmbeddingSize)

        # Convolutional Layers
        self.initConv = nn.Conv2d(inputChannels, layerChannels[0], 3, padding = 1)
        self.finalConv1 = nn.Conv2d(layerChannels[-1] // 2 + layerChannels[0], layerChannels[-1] // 2, 3, padding = 1)
        self.finalConv2 = nn.Conv2d(layerChannels[-1] // 2, outputChannels, 1)

        # UNET Layers
        layers = []

        for i in range(len(layerChannels)):
            layers.append(UNETLayer(layerUpscale[i], layerAttention[i], layerChannels[i], stepEmbeddingSize, classEmbeddingSize, numHeads))
        
        self.layers = nn.ModuleList(layers)

        self.layerUpscale = layerUpscale

    def forward(self, x, s, c):
        # First, get the encoding layers. We'll extend them across the batches.
        batches = x.shape[0]
        stepEmbeddings = self.stepEncoder(s).repeat(batches, 1)
        classEmbeddings = self.classEncoder(c[0])[None, :]

        for cClass in c[1:]:
            cEmbedding = self.classEncoder(cClass)[None, :]
            classEmbeddings = torch.cat([classEmbeddings, cEmbedding], dim = 0)

        # Initial Convolution Layer.
        x = F.relu(self.initConv(x))

        # Next, run the input through every layer.
        residualOutputs = []
        for i, layer in enumerate(self.layers):
            x, residual = layer(x, stepEmbeddings, classEmbeddings)

            if self.layerUpscale[i]:
                x = torch.cat([x, residualOutputs[-1]], dim = 1) # Residual Concats.
                residualOutputs = residualOutputs[:-1]

            else:
                residualOutputs.append(residual)

        # Finally, run it through the last two convolutional layers and output our result.
        return self.finalConv2(F.relu(self.finalConv1(x)))

In [8]:
# Create a new model.
model = UNET(
    layerChannels = [64, 128, 256, 512, 512, 384],
    layerAttention = [False, True, False, False, False, True],
    layerUpscale = [False, False, False, True, True, True],
    inputChannels = 3,
    outputChannels = 3,
    numHeads = 8,
    timeSteps = 1000,
    classCount = 10,
    stepEmbeddingSize = 32,
    classEmbeddingSize = 32
)

In [ ]:
# Training loop.
trainDataset = DataLoader(dataset, batch_size = 32, shuffle = True)

scheduler = NoiseScheduler()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

epochs = 100

for epoch in range(epochs):
    totalLoss = 0.0
    
    for batch in trainDataset:
        # Setting up the data.
        images, labels = batch
        timestamp = random.choice(range(1000))
        
        inputImages, expected = scheduler.add_noise(images, timestamp)
        
        # Forward pass.
        result = model(inputImages, timestamp, labels)
        loss = criterion(result, expected)
        totalLoss += loss.item()
        
        # Backward pass.
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    totalLoss /= len(trainDataset)
    print(f"Eopch #{epoch + 1}: {totalLoss}")

In [206]:
# Save the model.
torch.save(model, "models/Unconditional_Generation.pth")

In [208]:
# Load the model.
model = torch.load("models/Unconditional_Generation.pth", weights_only = False)

In [268]:
# We'll generate an image by progressively removing noise for the full 1000 timesteps.
def generate_image(modelRef, requestedClass):
    image = torch.rand(1, 3, 32, 32)
    scheduler = NoiseScheduler()
    
    currentStep = 999
    while currentStep > 0:
        predictedNoise = modelRef(image, currentStep, torch.tensor([requestedClass]))
        variance = ((1 - scheduler.cumulativeAlphas[currentStep - 1]) * scheduler.betas[currentStep]) / (1 - scheduler.cumulativeAlphas[currentStep])
        image = (1 / torch.sqrt(scheduler.alphas[currentStep])) * (image - scheduler.betas[currentStep] / (1 - scheduler.cumulativeAlphas[currentStep])) + torch.sqrt(variance) * torch.randn_like(image)

        currentStep -= 1
        
    return image[0]

# Displays a single image. Mainly used so that we can visually see the results.
def display_image(image):
    image = image.permute(1, 2, 0).numpy().clip(0, 1)
    plt.imshow(image)
    plt.axis("off")
    plt.show()